# Data Cleaning & Feature Engineering | NYC Airbnb Dataset

Taking the raw dataset from EDA findings and making deliberate decisions:
handling outliers properly, encoding categoricals, scaling numeric features,
and engineering new features that could improve downstream modeling.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


df = pd.read_csv('../data/AB_NYC_2019.csv')
print(f"Shape before baseline cleaning: {df.shape}")
# Baseline cleaning from notebook 01
df['reviews_per_month'] = df['reviews_per_month'].fillna(0)
df = df.dropna(subset=['name'])

print(f"Shape after baseline cleaning: {df.shape}")
df.head()

Shape before baseline cleaning: (48895, 16)
Shape after baseline cleaning: (48879, 16)


,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
0,2539,Clean & quiet apt home by the park,2787,John,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,2018-10-19,0.21,6,365
1,2595,Skylit Midtown Castle,2845,Jennifer,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,2019-05-21,0.38,2,355
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Elisabeth,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,NaN,0.00,1,365
3,3831,Cozy Entire Floor of Brownstone,4869,LisaRoxanne,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,1,270,2019-07-05,4.64,1,194
4,5022,Entire Apt: Spacious Studio/Loft by central park,7192,Laura,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,80,10,9,2018-11-19,0.10,1,0


## Handling Outliers | Deliberate Decisions, Not Blind Dropping

In [ ]:
# Price Outlier Decision

# From EDA: 6.1% of listings above $334 (IQR upper bound)
# Decision: these are legitimate high-end listings, not errors
# However $0 listings are likely data issues — inspect and remove

print(f"Zero-price listings: {len(df[df['price'] == 0])}")
print(f"Listings above $1000: {len(df[df['price'] > 1000])}")

# Remove $0 listings (no legitimate Airbnb listing is free)
df = df[df['price'] > 0]

# Cap extreme outliers at 99th percentile for modeling purposes
# (keeping original df intact, creating a model-ready version)
p99 = df['price'].quantile(0.99)
df_model = df.copy()
df_model['price_capped'] = df_model['price'].clip(upper=p99)

print(f"\n99th percentile price: ${p99:.2f}")
print(f"Shape after removing $0 listings: {df.shape}")

Zero-price listings: 11
Listings above $1000: 239

99th percentile price: $799.00
Shape after removing $0 listings: (48868, 16)


## Feature Engineering

In [5]:
# Date Based Feature 
# 'last_review' is a date column, convert and extract days since last review
df_model['last_review'] = pd.to_datetime(df_model['last_review'])

reference_date = pd.Timestamp('2019-12-31')  # dataset is from 2019
df_model['days_since_last_review'] = (reference_date - df_model['last_review']).dt.days

# Listings with no reviews get a large number (never reviewed)
df_model['days_since_last_review'] = df_model['days_since_last_review'].fillna(
    df_model['days_since_last_review'].max() + 1
)

print(df_model['days_since_last_review'].describe())

count    48868.000000
mean      1017.484264
std       1170.062953
min        176.000000
25%        195.000000
50%        362.000000
75%       1375.250000
max       3201.000000
Name: days_since_last_review, dtype: float64


In [6]:
# Text Based Feature

# Extract listing name length as a proxy for description effort
df_model['name_length'] = df_model['name'].fillna('').apply(len)

# Flag listings that mention keywords associated with higher prices
keywords = ['luxury', 'cozy', 'spacious', 'modern', 'stunning', 'beautiful']
pattern = '|'.join(keywords)
print(f"Pattern for premium keywords: {pattern}")
df_model['has_premium_keyword'] = df_model['name'].str.lower().str.contains(
    pattern, na=False).astype(int)

print(df_model[['name', 'name_length', 'has_premium_keyword']].head(10))

Pattern for premium keywords: luxury|cozy|spacious|modern|stunning|beautiful
                                               name  name_length  \
0                Clean & quiet apt home by the park           34   
1                             Skylit Midtown Castle           21   
2               THE VILLAGE OF HARLEM....NEW YORK !           35   
3                   Cozy Entire Floor of Brownstone           31   
4  Entire Apt: Spacious Studio/Loft by central park           48   
5         Large Cozy 1 BR Apartment In Midtown East           41   
6                                   BlissArtsSpace!           15   
7                  Large Furnished Room Near B'way            32   
8                Cozy Clean Guest Room - Family Apt           34   
9                Cute & Cozy Lower East Side 1 bdrm           34   

   has_premium_keyword  
0                    0  
1                    0  
2                    0  
3                    1  
4                    1  
5                    1  

In [7]:
# Availibity Features

# Bin availability_365 into categories
# A listing available <30 days is likely a secondary/occasional rental
# >300 days is likely a professional host

df_model['availability_category'] = pd.cut(
    df_model['availability_365'],
    bins=[-1, 30, 90, 180, 365],
    labels=['rarely_available', 'occasional', 'part_time', 'full_time']
)

df_model['availability_category'].value_counts()

availability_category
rarely_available    22791
full_time           14357
occasional           6438
part_time            5282
Name: count, dtype: int64

## Encoding Categorical Variables

In [8]:
 # One-hot encode room_type and neighbourhood_group

df_encoded = pd.get_dummies(df_model, 
                             columns=['room_type', 'neighbourhood_group'],
                             drop_first=True)  # avoid dummy variable trap

# Label encode availability_category (it has natural order)
availability_order = {'rarely_available': 0, 'occasional': 1, 
                      'part_time': 2, 'full_time': 3}
df_encoded['availability_encoded'] = df_model['availability_category'].map(
    availability_order)

print("New encoded columns:")
print([c for c in df_encoded.columns if 'room_type' in c 
       or 'neighbourhood' in c or 'availability' in c])

New encoded columns:
['neighbourhood', 'availability_365', 'availability_category', 'room_type_Private room', 'room_type_Shared room', 'neighbourhood_group_Brooklyn', 'neighbourhood_group_Manhattan', 'neighbourhood_group_Queens', 'neighbourhood_group_Staten Island', 'availability_encoded']


## Scaling Numeric Features

In [10]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

numeric_features = ['minimum_nights', 'number_of_reviews', 
                    'reviews_per_month', 'days_since_last_review',
                    'name_length', 'availability_365']

# StandardScaler: mean=0, std=1 — good for most ML models
scaler = StandardScaler()
df_scaled = df_encoded.copy()
df_scaled[numeric_features] = scaler.fit_transform(
    df_encoded[numeric_features])

df_scaled[numeric_features].describe().round(2)

,minimum_nights,number_of_reviews,reviews_per_month,days_since_last_review,name_length,availability_365
count,48868.0,48868.00,48868.00,48868.00,48868.00,48868.00
mean,-0.0,0.00,0.00,0.00,0.00,-0.00
std,1.0,1.00,1.00,1.00,1.00,1.00
min,-0.3,-0.52,-0.68,-0.72,-3.42,-0.86
25%,-0.3,-0.50,-0.66,-0.70,-0.56,-0.86
50%,-0.2,-0.41,-0.45,-0.56,0.01,-0.52
75%,-0.1,0.02,0.31,0.31,0.87,0.87
max,62.1,13.59,35.94,1.87,13.54,1.92


## Summary Decisions Made in This Notebook

- Removed $0-price listings (likely data errors, no legitimate listing is free)
- Retained high-end outliers (>$334) as legitimate market data; capped at 99th 
  percentile only for a model-ready copy, preserving the original
- Engineered `days_since_last_review` from date column, recency signal
- Engineered `name_length` and `has_premium_keyword` from listing name text
- Binned `availability_365` into interpretable categories
- One-hot encoded nominal categoricals (room_type, neighbourhood_group)
- Standard-scaled numeric features for model readiness

**Key distinction practiced here**: outlier decisions should be deliberate 
and documented, not automatic. Dropping data always has a cost, the goal 
is to make that tradeoff consciously.

**Next**: `04_data_visualization.ipynb` : polished, multi-variable visualizations 
going deeper than EDA plots.